In [1]:
import jax
import jax.numpy as jnp

In [2]:
a1 = jnp.array([1.,2.,3.])
a2 = jnp.array([3.,5.,6.])
a3 = jnp.array([2.,7.,4.])
a4 = jnp.array([2.,8.,7.])

A1 = jnp.outer(a1, a1)
A2 = jnp.outer(a2, a2)
A3 = jnp.outer(a3, a3)
A4 = jnp.outer(a4, a4)

A = A1+A2+A3

A

Array([[14., 31., 29.],
       [31., 78., 64.],
       [29., 64., 61.]], dtype=float32)

In [3]:
assert jnp.all(jnp.linalg.eigvalsh(A) > 0), "Not PD"
assert jnp.all(A == A.T), "Not Symmetric"

In [4]:
L = jnp.linalg.cholesky(A)
R = jnp.linalg.cholesky(A, upper=True)

R.T@R.T, jnp.abs(R.T@R - A)

(Array([[14.000001  ,  0.        ,  0.        ],
        [56.343666  ,  9.35715   ,  0.        ],
        [35.868504  , -0.28160676,  0.9236643 ]], dtype=float32),
 Array([[9.5367432e-07, 0.0000000e+00, 0.0000000e+00],
        [0.0000000e+00, 0.0000000e+00, 3.8146973e-06],
        [0.0000000e+00, 3.8146973e-06, 0.0000000e+00]], dtype=float32))

In [5]:
Rp = jax.lax.linalg.cholesky_update(R, a4)

R2 = jnp.linalg.cholesky(A+A4, upper=True)

Rp,R2

(Array([[ 4.2426410e+00,  1.1078006e+01,  1.0135198e+01],
        [ 0.0000000e+00,  4.3906474e+00,  1.7587900e+00],
        [ 0.0000000e+00, -4.6474074e-08,  2.0455894e+00]], dtype=float32),
 Array([[ 4.2426405, 11.078007 , 10.135198 ],
        [ 0.       ,  4.3906455,  1.7587875],
        [ 0.       ,  0.       ,  2.0455892]], dtype=float32))

In [6]:
"""
Efficient rank-1 Cholesky downdate in JAX (Stewart-style / cholupdate-style)

Implements an O(n^2) in-place-like downdate for lower-triangular Cholesky factors.

Given L (lower-triangular, positive diagonal) with A = L @ L.T, and a vector v,
compute L_new such that L_new @ L_new.T = A - v v^T, when this remains SPD.

Also exposes a symmetric API for rank-1 update for completeness.

Notes
-----
* The algorithm mirrors the classic cholupdate/downdate scheme using plane
  rotations (Givens) applied to the (k,k) pivot and the working vector.
* We include a positive-definiteness guard for downdates based on
  rho^2 = 1 - ||solve(L, v)||^2 > 0. If this fails, we return (L, False).
* Written with JAX transforms in mind (jit-friendly, no Python loops after jitting).

References
----------
- Stewart (dchud/dchdd); Seeger (2008) "Low Rank Updates for the Cholesky Decomposition".
"""
from __future__ import annotations
from typing import Tuple
import jax
import jax.numpy as jnp
import jax.scipy as jsp
from jax import lax

Array = jax.Array


def _rank1_modify_lower(L: Array, x: Array, sign: float) -> Tuple[Array, Array, Array]:
    """Core O(n^2) rank-1 modify (update/downdate) for LOWER-triangular L.

    JIT-friendly: avoids dynamic-length slicing inside the loop. We do full-width
    vector updates gated by a boolean mask (idx > k) so shapes remain static.
    """
    n = L.shape[0]

    def body(k, carry):
        Lk, xk, ok = carry
        Lkk = Lk[k, k]
        xk_k = xk[k]
        # r^2 = Lkk^2 +/- xk_k^2
        r2 = Lkk * Lkk + sign * (xk_k * xk_k)
        ok_k = (r2 > 0.0) | (sign > 0.0)  # only relevant for downdate
        r = jnp.sqrt(jnp.maximum(r2, 0.0))
        # Givens parameters
        c = jnp.where(Lkk != 0.0, r / Lkk, 1.0)
        s = jnp.where(Lkk != 0.0, xk_k / Lkk, 0.0)
        # Set the pivot
        Lk = Lk.at[k, k].set(r)
        # Vectorized update for rows j > k
        idx = jnp.arange(n)
        mask = idx > k
        Lcol = Lk[:, k]
        Lcol_new = jnp.where(mask, (Lcol + sign * s * xk) / c, Lcol)
        # Ensure diagonal just set remains r
        Lcol_new = Lcol_new.at[k].set(r)
        Lk = Lk.at[:, k].set(Lcol_new)
        # Update x: x_j <- c * x_j - s * L_{j,k}^{new} for j > k; x_k <- 0
        x_new = jnp.where(mask, c * xk - s * Lcol_new, xk)
        x_new = x_new.at[k].set(0.0)
        ok = ok & ok_k
        return (Lk, x_new, ok)

    L_out, x_out, ok = lax.fori_loop(0, n, body, (L, x, jnp.array(True)))
    return L_out, x_out, ok


@jax.jit
def cholesky_update_lower(L: Array, v: Array) -> Array:
    """Rank-1 update: return L' s.t. L' L'^T = (L L^T) + v v^T.

    Parameters
    ----------
    L : (n, n) lower-triangular Cholesky factor (diag>0)
    v : (n,) update vector
    """
    L_new, _, _ = _rank1_modify_lower(L, v, sign=+1.0)
    # Ensure positive diagonal (guard against round-off)
    diag = jnp.sign(jnp.diag(L_new))
    diag = jnp.where(diag == 0.0, 1.0, diag)
    L_new = L_new * diag[:, None]
    return L_new


from functools import partial

@partial(jax.jit, static_argnames=("check_pd",))
def cholesky_downdate_lower(L: Array, v: Array, check_pd: bool = True) -> Tuple[Array, Array]:
    """Rank-1 downdate: return (L', ok) with L' L'^T = (L L^T) - v v^T when ok=True.

    JIT notes
    ---------
    * `check_pd` is a **static** Python bool (compiled into the XLA program).
      Pass a Python literal True/False; do not pass a JAX boolean array.
    * SPD guard uses jax.scipy's `solve_triangular` with RHS shape (n,),
      which sidesteps shape pitfalls.
    """
    if check_pd:
        # Pre-check: rho^2 = 1 - ||L^{-1} v||^2 > 0
        p = jsp.linalg.solve_triangular(L, v, lower=True, trans='N')
        rho2 = 1.0 - jnp.dot(p, p)
        ok0 = rho2 > 0.0
    else:
        ok0 = jnp.array(True)

    L_new, _, ok_alg = _rank1_modify_lower(L, v, sign=-1.0)
    ok = ok0 & ok_alg

    # Enforce positive diagonal convention without materializing a dense D
    diag = jnp.sign(jnp.diag(L_new))
    diag = jnp.where(diag == 0.0, 1.0, diag)
    L_new = L_new * diag[:, None]
    return L_new, ok


# Sequential series of up-/downdates on a single factor (mixed signs)
from functools import partial as __partial

@__partial(jax.jit, static_argnames=("check_pd",))
def cholesky_modify_lower_series(L: Array, V: Array, signs: Array, check_pd: bool = True) -> Tuple[Array, Array, Array]:
    """Apply a sequence of rank-1 up-/downdates to a single lower-triangular factor.

    Parameters
    ----------
    L     : (n, n) lower-triangular Cholesky factor (diag>0)
    V     : (m, n) sequence of vectors v_k
    signs : (m,)   +1.0 for update (A + v v^T), -1.0 for downdate (A - v v^T)
    check_pd : if True, guard each downdate step k with rho^2_k = 1 - ||L_k^{-1} v_k||^2 > 0.

    Returns
    -------
    L_final : (n, n) factor after applying the *valid* prefix of operations
    ok_all  : scalar bool; True iff all steps were valid
    ok_each : (m,) bool flags per step (True if step was valid wrt SPD + algebra)

    Notes
    -----
    * If a step is invalid (downdate would break SPD), it is skipped and all subsequent
      steps are also skipped; `ok_each[k]` marks the first failure.
    * This function is JIT-friendly (static shapes, `lax.scan`).
    """
    signs = signs.astype(V.dtype)

    def scan_step(carry, inp):
        Lk, ok_so_far = carry
        v_k, s_k = inp

        if check_pd:
            def _guard_true(_):
                p = jsp.linalg.solve_triangular(Lk, v_k, lower=True, trans='N')
                rho2 = 1.0 - jnp.dot(p, p)
                return rho2 > 0.0
            ok_guard = lax.cond(s_k < 0.0, _guard_true, lambda _: jnp.array(True), operand=None)
        else:
            ok_guard = jnp.array(True)

        L_new, _, ok_alg = _rank1_modify_lower(Lk, v_k, sign=s_k)
        # Keep positive diagonal convention
        diag = jnp.sign(jnp.diag(L_new))
        diag = jnp.where(diag == 0.0, 1.0, diag)
        L_new = L_new * diag[:, None]

        ok_step = ok_guard & ok_alg
        apply = ok_so_far & ok_step
        L_next = lax.select(apply, L_new, Lk)
        ok_next = ok_so_far & ok_step
        return (L_next, ok_next), ok_step

    (L_final, ok_all), ok_each = lax.scan(scan_step, (L, jnp.array(True)), (V, signs))
    return L_final, ok_all, ok_each

@jax.jit
def cholesky_update_lower_series(L: Array, V: Array) -> Tuple[Array, Array, Array]:
    """Sequence of rank-1 **updates** (sign=+1) on a single factor.

    Returns (L_final, ok_all, ok_each); all ok flags should be True for updates.
    """
    m = V.shape[0]
    signs = jnp.ones((m,), dtype=V.dtype)
    return cholesky_modify_lower_series(L, V, signs, check_pd=False)

@__partial(jax.jit, static_argnames=("check_pd",))
def cholesky_downdate_lower_series(L: Array, V: Array, check_pd: bool = True) -> Tuple[Array, Array, Array]:
    """Sequence of rank-1 **downdates** (sign=-1) on a single factor.

    Returns (L_final, ok_all, ok_each) with SPD guards per step if `check_pd=True`.
    """
    m = V.shape[0]
    signs = -jnp.ones((m,), dtype=V.dtype)
    return cholesky_modify_lower_series(L, V, signs, check_pd=check_pd)


In [7]:
jnp.max(jnp.abs( cholesky_update_lower(L, a4) - R2.T ))

Array(1.9073486e-06, dtype=float32)

In [8]:
Lp, ok = cholesky_downdate_lower(cholesky_update_lower(L, a4), a4, check_pd=True)
assert ok, "downdating failed!"

jnp.max(jnp.abs( Lp - L )) 

Array(6.631017e-07, dtype=float32)

In [9]:
from matfree import decomp, funm

def assert_pd(M):
    assert jnp.all(jnp.linalg.eigvals(M) > 0), "Not PD!"
    
def assert_sym(M):
    assert jnp.all(M == M.T) or jnp.all(jnp.isclose(M, M.T, atol=1e-7)), "Not symmmetric!"
    
def assert_spd(M):
    assert_sym(M)
    assert_pd(M)

In [66]:
from jax.flatten_util import ravel_pytree

# ======== Clean subspace-augmentation + re-tridiagonalization (no fori_loop) ========

def _orth_against(Q: jax.Array, Y: jax.Array, eps: float = 1e-12) -> jax.Array:
    # Orthonormal basis for (I-QQ^T)Y
    Yp = Y - Q @ (Q.T @ Y)
    if Yp.size == 0:
        return Yp
    U, R = jnp.linalg.qr(Yp, mode="reduced")
    if R.size == 0:
        return U[:, :0]
    keep = jnp.abs(jnp.diag(R)) > eps * jnp.linalg.norm(R, ord=2)
    return U[:, keep]


def _build_Z_or_R(Q_or_U, wt_apply):
    YQ_or_U = jax.vmap(wt_apply, in_axes=1, out_axes=1)(Q_or_U)     # (k, m)
    Mat  = YQ_or_U.T                                             # (m, k)
    return Mat


def augment_and_retridiag(Q: jax.Array,
                          T: jax.Array,
                          w_apply, 
                          wt_apply,
                          v0=None,
                          sign: float = +1.0,
                          mode: str = "exact",
                          probes: int = 0,
                          key = jax.random.PRNGKey(0)):
    """
    Given A ≈ Q T Q^T and a low-rank ± W W^T (via oracles w_apply, wt_apply),
    return (Q_tilde, T_tilde) with T_tilde tridiagonal so that
      A' ≈ Q_tilde T_tilde Q_tilde^T.
    - mode="exact": spans range(W) exactly by applying W to I_k (needs latent dim k).
    - mode="random": spans (I-QQ^T)range(W) via 'probes' random latent vectors.
    """
    m = Q.shape[1]
    D = Q.shape[0]
    # k_latent = _infer_k(wt_apply, D) # TODO compute once and pass from outside
    vflat,unravel_fn = ravel_pytree(wt_apply(jnp.zeros((D,))))
    k_latent = int(vflat.shape[0])
    
    # Z = Q^T W from W^T only
    Z = _build_Z_or_R(Q, wt_apply).squeeze()

    # Build residual basis U for (I-QQ^T)range(W)
    if mode == "exact":
        I_k = jnp.eye(k_latent, dtype=Q.dtype) # ! Problem.
        Y   = jax.vmap(lambda v: w_apply(unravel_fn(v)), in_axes=1, out_axes=1)(I_k)  # W * I_k  (D, k)
        U   = _orth_against(Q, Y)
    else:
        s   = probes if probes > 0 else min(k_latent, 6)     # a few probes by default
        Omega = jax.random.normal(key, (k_latent, s))
        Y     = jax.vmap(lambda v: w_apply(unravel_fn(v)), in_axes=1, out_axes=1)(Omega)  # (D, s)
        U     = _orth_against(Q, Y)

    r = 0 if U.size == 0 else U.shape[1]
    R = _build_Z_or_R(U, wt_apply).squeeze()

    # Assemble small augmented matrix \bar T
    T11 = T + sign * (Z.T @ Z)
    T12 = sign * (Z.T @ R)
    T22 = sign * (R.T @ R)
    Tbar = jnp.block([[T11, T12],
                      [T12.T, T22]])           # (m+r, m+r)
    # ? Old, did not work for my GGN,W,WT operators
    # T11 = T + sign * (Z @ Z.T)
    # T12 = sign * (Z @ R.T)
    # T22 = sign * (R @ R.T)
    # Tbar = jnp.block([[T11, T12],
    #                   [T12.T, T22]])           # (m+r, m+r)

    # Re-tridiagonalize \bar T using matfree's Lanczos on the small operator
    n_small = Tbar.shape[0]
    tridiag_small = decomp.tridiag_sym(n_small)
    def matvec_small(g):  # R^{m+r} -> R^{m+r}
        return Tbar @ g
    v0_small = jax.random.normal(key, (n_small,))
    res_small = tridiag_small(matvec_small, v0_small)
    Q_small   = res_small.Q_tall     # (m+r, m+r) orthogonal
    J_small   = res_small.J_small    # (m+r, m+r) tridiagonal

    Qbar   = Q if r == 0 else jnp.concatenate([Q, U], axis=1)  # (D, m+r)
    Qtilde = Qbar @ Q_small                                     # (D, m+r)
    Ttilde = J_small
    return Qtilde, Ttilde

In [103]:
# =============================== quick tests ===============================
def w_apply_mat(w_apply, Y):
    return jax.vmap(w_apply, in_axes=1, out_axes=1)(Y) if Y.ndim == 2 else w_apply(Y)

def wt_apply_mat(wt_apply, X):
    return jax.vmap(wt_apply, in_axes=1, out_axes=1)(X) if X.ndim == 2 else wt_apply(X)


_key = lambda i: jax.random.PRNGKey(i)

D = 1_000
D2 = 200
M = 200

I_D = jnp.eye(D)
I_M = jnp.eye(M)

B = jax.random.normal(_key(1), (D, D2))
X = B@B.T

def matvec(v):
    return X @ v
    
# assert_spd(X)


matfun = funm.dense_funm_sym_eigh(jnp.log)
tridiag = decomp.tridiag_sym(M)

v = jax.random.normal(_key(5), (D,))

res = tridiag(matvec, v)
Q_or_U = res.Q_tall
T = res.J_small

k = 100
W = jax.random.normal(_key(123), (D, k)) # * 0.4
w_apply  = lambda y: W @ y             # R^k -> R^D
wt_apply = lambda x: W.T @ x           # R^D -> R^k

print("\n[BASELINE] A ≈ Q T Qᵀ")
rel_err_baseline = jnp.linalg.norm(Q_or_U@T@Q_or_U.T - X) / jnp.linalg.norm(X)
print("  relative Frobenius error  :", float(rel_err_baseline))


# UPDATE: A' = X + W W^T
Q_upd, T_upd = augment_and_retridiag(Q_or_U, T, w_apply, wt_apply, sign=+1.0, mode="exact", key=_key(10))
A_true_plus   = X + W @ W.T
A_approx_plus = Q_upd @ T_upd @ Q_upd.T

rel_err_plus = jnp.linalg.norm(A_approx_plus - A_true_plus) / jnp.linalg.norm(A_true_plus)
orth_err_upd = jnp.linalg.norm(Q_upd.T @ Q_upd - jnp.eye(Q_upd.shape[1]))
offband = lambda M: jnp.linalg.norm(M - jnp.triu(jnp.tril(M, 1), -1))

print("\n[UPDATE] A + W Wᵀ ≈ Q̃ T̃ Q̃ᵀ")
print("  Q̃ shape, T̃ shape        :", Q_upd.shape, T_upd.shape)
print("  ||Q̃ᵀ Q̃ - I||_F          :", float(orth_err_upd))
print("  ||off-band(T̃)||_F        :", float(offband(T_upd)))
print("  relative Frobenius error  :", float(rel_err_plus))

# DOWNDATE: use a small W so SPD is preserved
W_small = W # * 0.1
w_apply_s  = lambda y: W_small @ y
wt_apply_s = lambda x: W_small.T @ x

Q_dwn, T_dwn = augment_and_retridiag(Q_or_U, T, w_apply_s, wt_apply_s, sign=-1.0, mode="exact", key=_key(11))
# Q_dwn, T_dwn = augment_and_retridiag(Q, T, w_apply_s, wt_apply_s, sign=-1.0, mode="random", key=_key(11), probes=100)
A_true_minus   = X - W_small @ W_small.T
A_approx_minus = Q_dwn @ T_dwn @ Q_dwn.T

rel_err_minus = jnp.linalg.norm(A_approx_minus - A_true_minus) / jnp.linalg.norm(A_true_minus)
orth_err_dwn  = jnp.linalg.norm(Q_dwn.T @ Q_dwn - jnp.eye(Q_dwn.shape[1]))

print("\n[DOWNDATE] A - W Wᵀ ≈ Q̃ T̃ Q̃ᵀ")
print("  Q̃ shape, T̃ shape        :", Q_dwn.shape, T_dwn.shape)
print("  ||Q̃ᵀ Q̃ - I||_F          :", float(orth_err_dwn))
print("  ||off-band(T̃)||_F        :", float(offband(T_dwn)))
print("  relative Frobenius error  :", float(rel_err_minus))

import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 3, figsize=(20,5))

pl0 = axs[0].imshow(jnp.abs(X - Q_or_U@T@Q_or_U.T))
plt.colorbar(pl0)
axs[0].set_title("Baseline approx vs. True")

pl1 = axs[1].imshow(jnp.abs(A_true_plus - A_approx_plus))
plt.colorbar(pl1)
axs[1].set_title("LR Updated vs. True")

pl2 = axs[2].imshow(jnp.abs(A_true_minus - A_approx_minus))
plt.colorbar(pl2)
axs[2].set_title("LR Downdated vs. True")

plt.tight_layout()
plt.show()


[BASELINE] A ≈ Q T Qᵀ
  relative Frobenius error  : 0.19697020947933197


TypeError: add got incompatible shapes for broadcasting: (200, 200), (100, 100).

In [62]:
jnp.linalg.slogdet(A_true_minus)[1].item(), \
jnp.linalg.slogdet(A_approx_minus)[1].item()

(-3848.1689453125, -4077.400390625)

In [44]:
jnp.linalg.slogdet(A_true_plus)[1].item(), \
jnp.linalg.slogdet(A_approx_plus)[1].item()

(-4370.54248046875, -4553.37353515625)

In [45]:
jnp.linalg.slogdet(X)[1].item(), \
jnp.linalg.slogdet(Q_or_U@T@Q_or_U.T)[1].item()

(-6110.92578125, -6303.60595703125)

## Tests with GGN

In [68]:
# load MAP state

import optax

from src.utils import load_checkpoint, load_yaml, count_model_params
from src.scalemodels import TrainState, EMPTY_STATS
from src.toymodels import SimpleClassifier
from src.toydata import get_dataloaders

model_name = 'toyclassifier_banana'
cfg_path = f'config/toy/{model_name}.yml'
dataset = 'banana'

cfg = load_yaml(cfg_path)
model_cfg = cfg['model']
opt_cfg = cfg['optimization']
alpha = opt_cfg["alpha"]
map_cfg = opt_cfg["map"]

model_type = model_cfg.get("name", "regressor")  # 'regressor' or 'classifier'
num_h = model_cfg["num_h"]
num_l = model_cfg["num_l"]
num_c = model_cfg.get("num_c", 2) if model_type == "classifier" else 1
rng_model = jax.random.PRNGKey(model_cfg["seed"])
map_batch_size = map_cfg["batch_size"]
epochs_map = map_cfg["epochs"]
lr_map = map_cfg["lr"]

model = SimpleClassifier(numh=num_h, numl=num_l, numc=num_c)

train_loader, test_loader, _ = get_dataloaders(dataset=dataset, batch_size=map_batch_size)

dummy_input = next(iter(train_loader))[0][:1]
variables = model.init(rng_model, dummy_input)
optimizer_map = optax.adam(1e-3)
model_state = TrainState.create(
    apply_fn=model.apply,
    params=variables['params'],
    tx=optimizer_map,
    batch_stats = variables.get('batch_stats', EMPTY_STATS),
)
map_ckpt_prefix = f"map_{dataset}"

map_state = load_checkpoint(
    ckpt_dir="checkpoint/map/",
    prefix=map_ckpt_prefix,
    target=model_state
)

D = count_model_params(variables)

[checkpoint] Loaded model checkpoint from /Users/nielsraunkjaer/Desktop/thesis/laplace-inducing-points/checkpoint/map (prefix=map_banana)


In [75]:
# compute W and GGN

from src.ggn import compute_W_vps, compute_ggn_vp, compute_ggn_dense
from src.utils import flatten_nn_params

flat_params, unravel_fn = flatten_nn_params(map_state.params)

FULL_DATA  = next(iter(train_loader))[0] # 32 samples
GGN_DATA   = FULL_DATA[:-1]
EXTRA_TERM = FULL_DATA[None,-1]

GGN_full_dense, *_ =  compute_ggn_dense(map_state, FULL_DATA, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
GGN_dense, *_ =  compute_ggn_dense(map_state, GGN_DATA, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)

GGN_full =  compute_ggn_vp(map_state, FULL_DATA, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
GGN      =  compute_ggn_vp(map_state, GGN_DATA, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
W, WT    =  compute_W_vps(map_state, GGN_DATA, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)
Wp, WTp  =  compute_W_vps(map_state, EXTRA_TERM, model_type=model_type, flat_params=flat_params, unravel_fn=unravel_fn, full_set_size=None)

### <font color='orange'> TODOs </font>

- ~~understand and test new sampling algorithm from Niko~~
- fix shape stuff for W_/WT_apply
- look into $\text{``Optimal Stochastic Trace Estimation in Generative Modeling''}$

In [94]:
eps = jax.random.normal(_key(123), (D,))

tridiag = decomp.tridiag_sym(32)

v = jax.random.normal(_key(5), (D,))

res = tridiag(GGN, v)
Q_or_U = res.Q_tall
T = res.J_small

Q_dwn, T_dwn = augment_and_retridiag(Q_or_U, T, Wp, WTp, v0=v, sign=-1.0, mode="exact", key=_key(11))

In [95]:
jnp.linalg.slogdet(GGN_dense)

SlogdetResult(sign=Array(-1., dtype=float32), logabsdet=Array(-10503.33, dtype=float32))

In [96]:
jnp.linalg.slogdet(Q_dwn@T_dwn@Q_dwn.T)

SlogdetResult(sign=Array(1., dtype=float32), logabsdet=Array(-10217.684, dtype=float32))